# 06 Spark-Structured-Streaming: Kafka zu Parquet

## Zweck
Open-Meteo-Ereignisse zur Luftqualität mit Spark-Structured-Streaming einlesen, den expliziten Ereignisvertrag aus Phase 5 validieren, gültige Ereignisse mit statischen Stadtdaten anreichern und als Parquet-Dateien speichern.

Der bevorzugte Nachweis in der FH-Umgebung verwendet Kafka. Falls Kafka oder der Spark-Kafka-Konnektor nicht verfügbar ist und der Fallback ausdrücklich erlaubt wurde, nutzt das Notebook dieselben Spark-Transformationen mit einer lokalen JSONL-Dateiquelle. Der Fallback weist nur die lokale Funktionsfähigkeit nach, nicht das Einlesen aus Kafka.

## Eingaben

- Über `.env` konfiguriertes Kafka-Topic oder lokale JSONL-Ereignisse aus Phase 5 als expliziter Fallback
- `data/silver/city_reference.parquet`
- `data/silver/city_metadata.parquet`

## Ausgaben

- `data/bronze/open_meteo_stream/`
- `data/bronze/open_meteo_stream_rejects/`
- `data/silver/open_meteo_city_hourly/`
- `data/gold/live_air_quality_latest.parquet/`
- `data/checkpoints/open_meteo_stream_*/`

## Verwendete Technologien
PySpark, Spark-Structured-Streaming, Kafka-Quellkonnektor, expliziter `StructType`, Verknüpfungen zwischen statischen Daten und Streams, Parquet und Checkpoints.

## Konfiguration

Für den verbindlichen Nachweis in Docker oder der FH werden `SPARK_KAFKA_MODE=kafka` und `ALLOW_SPARK_KAFKA_MOCK_FALLBACK=false` gesetzt. Docker-Jupyter unter `http://localhost:8888` lädt den Spark-Kafka-Connector automatisch über `.env.docker.example`.

Ein direkt auf dem Host gestarteter lokaler Jupyter-Kernel verwendet `.env` mit `SPARK_KAFKA_MODE=auto`. Fehlt dort der Spark-Kafka-Connector, wechselt das Notebook transparent auf den lokalen Spark-Dateistream. Dieser Host-Fallback prüft die Transformationen, ist aber kein Kafka-Nachweis.

`SPARK_MASTER_URL` wird aus `.env` gelesen. Lokale Parquet-Ausgaben sind mit `local[*]` zuverlässig. Ein entfernter Spark-Master setzt voraus, dass `DATA_DIR` und `CHECKPOINT_DIR` für die Worker sichtbar sind.


### Dateipfade für Phase 6 bestimmen

Alle Bronze-, Silver-, Gold- und Checkpoint-Pfade werden aus dem Repository-Stammverzeichnis abgeleitet. Dadurch werden versehentliche Schreibvorgänge unter `notebooks/data/` vermieden.

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os
import shutil
import socket

_cwd = Path.cwd().resolve()
_candidate_root = _cwd.parent if _cwd.name == "notebooks" else _cwd
load_dotenv(_candidate_root / ".env")

_env_root = os.getenv("PROJECT_ROOT")
PROJECT_ROOT = Path(_env_root).resolve() if _env_root else _candidate_root

def project_path(env_name: str, default: str) -> Path:
    path = Path(os.getenv(env_name, default))
    return path if path.is_absolute() else PROJECT_ROOT / path

DATA_DIR = project_path("DATA_DIR", "data")
CHECKPOINT_DIR = project_path("CHECKPOINT_DIR", "data/checkpoints")
CITY_REFERENCE_PATH = DATA_DIR / "silver" / "city_reference.parquet"
CITY_METADATA_PATH = DATA_DIR / "silver" / "city_metadata.parquet"
PHASE5_EVENTS_PATH = DATA_DIR / "bronze" / "open_meteo_raw" / "open_meteo_air_quality_events.jsonl"
PHASE5_SAMPLE_EVENTS_PATH = DATA_DIR / "samples" / "open_meteo_phase5_events_sample.jsonl"

BRONZE_STREAM_PATH = DATA_DIR / "bronze" / "open_meteo_stream"
REJECTS_STREAM_PATH = DATA_DIR / "bronze" / "open_meteo_stream_rejects"
SILVER_STREAM_PATH = DATA_DIR / "silver" / "open_meteo_city_hourly"
LATEST_SNAPSHOT_PATH = DATA_DIR / "gold" / "live_air_quality_latest.parquet"
MOCK_INPUT_DIR = DATA_DIR / "bronze" / "open_meteo_stream_mock_input"

CHECKPOINT_BRONZE = CHECKPOINT_DIR / "open_meteo_stream_bronze"
CHECKPOINT_REJECTS = CHECKPOINT_DIR / "open_meteo_stream_rejects"
CHECKPOINT_SILVER = CHECKPOINT_DIR / "open_meteo_stream_silver"

print({"project_root": str(PROJECT_ROOT), "data_dir": str(DATA_DIR), "city_reference_exists": CITY_REFERENCE_PATH.exists(), "city_metadata_exists": CITY_METADATA_PATH.exists()})


{'project_root': 'C:\\dev\\euro-air-quality-pipeline', 'data_dir': 'C:\\dev\\euro-air-quality-pipeline\\data', 'city_reference_exists': True, 'city_metadata_exists': True}


### Spark- und Kafka-Einstellungen laden

Diese Umgebungswerte wählen lokale oder FH-Spark-Ausführung, Broker-Topic und erlaubtes Fallback-Verhalten. Platzhalter-Topics sind sicher, solange keine Veröffentlichung aktiviert ist.

In [2]:
EXECUTION_ENV = os.getenv("EXECUTION_ENV", "local_project")
SPARK_MASTER_URL = os.getenv("SPARK_MASTER_URL", "local[*]")
KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "<kafka-host>:9092")
KAFKA_TOPIC = os.getenv("KAFKA_TOPIC_AIR_QUALITY_LIVE", "LIVE-bdeng_gXX_air_quality_live")
SPARK_KAFKA_MODE = os.getenv("SPARK_KAFKA_MODE", os.getenv("KAFKA_MODE", "auto")).lower()
ALLOW_SPARK_KAFKA_MOCK_FALLBACK = os.getenv("ALLOW_SPARK_KAFKA_MOCK_FALLBACK", "true").lower() == "true"
SPARK_KAFKA_CONNECTOR_PACKAGE = os.getenv("SPARK_KAFKA_CONNECTOR_PACKAGE", "").strip()
SPARK_SQL_SHUFFLE_PARTITIONS = int(os.getenv("SPARK_SQL_SHUFFLE_PARTITIONS", "8"))
ALLOW_SHARED_SPARK_STORAGE = os.getenv("ALLOW_SHARED_SPARK_STORAGE", "false").lower() == "true"
RESET_PHASE6_CHECKPOINTS = os.getenv("RESET_PHASE6_CHECKPOINTS", "false").lower() == "true"

if SPARK_MASTER_URL.startswith("spark://") and not ALLOW_SHARED_SPARK_STORAGE:
    print(
        "HINWEIS: Entfernter Spark-Master ohne bestaetigten gemeinsamen Speicher konfiguriert. "
        "Für Phase-6-Parquet-Ausgabe wird local[*] verwendet, "
        "solange kein gemeinsamer Storage für Driver und Worker nachgewiesen ist."
    )
    EFFECTIVE_SPARK_MASTER_URL = "local[*]"
else:
    EFFECTIVE_SPARK_MASTER_URL = SPARK_MASTER_URL

assert SPARK_KAFKA_MODE in {"auto", "kafka", "mock"}, (
    f"SPARK_KAFKA_MODE muss auto, kafka oder mock sein; erhalten: {SPARK_KAFKA_MODE!r}"
)
for required_path in [CITY_REFERENCE_PATH, CITY_METADATA_PATH]:
    assert required_path.exists(), f"Abhängigkeit aus vorheriger Phase fehlt: {required_path}"

if RESET_PHASE6_CHECKPOINTS:
    shutil.rmtree(CHECKPOINT_BRONZE, ignore_errors=True)
    shutil.rmtree(CHECKPOINT_REJECTS, ignore_errors=True)
    shutil.rmtree(CHECKPOINT_SILVER, ignore_errors=True)
    print("Checkpoints zurückgesetzt (RESET_PHASE6_CHECKPOINTS=true).")

print({
    "project_root": str(PROJECT_ROOT),
    "execution_env": EXECUTION_ENV,
    "spark_master_url": SPARK_MASTER_URL,
    "effective_spark_master": EFFECTIVE_SPARK_MASTER_URL,
    "allow_shared_spark_storage": ALLOW_SHARED_SPARK_STORAGE,
    "spark_sql_shuffle_partitions": SPARK_SQL_SHUFFLE_PARTITIONS,
    "spark_kafka_mode": SPARK_KAFKA_MODE,
    "kafka_bootstrap_servers": KAFKA_BOOTSTRAP_SERVERS,
    "kafka_topic": KAFKA_TOPIC,
    "data_dir": str(DATA_DIR),
    "checkpoint_dir": str(CHECKPOINT_DIR),
    "reset_checkpoints": RESET_PHASE6_CHECKPOINTS,
})


{'project_root': 'C:\\dev\\euro-air-quality-pipeline', 'execution_env': 'local_project', 'spark_master_url': 'local[*]', 'effective_spark_master': 'local[*]', 'allow_shared_spark_storage': False, 'spark_sql_shuffle_partitions': 8, 'spark_kafka_mode': 'auto', 'kafka_bootstrap_servers': '<kafka-host>:9092', 'kafka_topic': 'LIVE-bdeng_gXX_air_quality_live', 'data_dir': 'C:\\dev\\euro-air-quality-pipeline\\data', 'checkpoint_dir': 'C:\\dev\\euro-air-quality-pipeline\\data\\checkpoints', 'reset_checkpoints': False}


## Umsetzung

### Spark starten und Streaming-Quelle auswählen

Der Kafka-Pfad verwendet `readStream.format("kafka")`. Der lokale Fallback bleibt bewusst ein Spark-Streaming-Pfad: Die validierte JSONL-Datei aus Phase 5 wird in ein lokales Eingabeverzeichnis kopiert und mit `readStream.text()` gelesen.

#### Spark-Sitzung starten

Der Builder verwendet den konfigurierten Master. Falls JupyterHub den Kafka-Konnektor nicht vorinstalliert hat, kann optional ein Konnektor-Paket angegeben werden.

In [3]:
try:
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import (
        col, current_timestamp, desc, from_json, lit, row_number, to_timestamp, when
    )
    from pyspark.sql.types import DoubleType, StringType, StructField, StructType
    from pyspark.sql.window import Window
    HAS_PYSPARK = True
except ModuleNotFoundError:
    HAS_PYSPARK = False

if HAS_PYSPARK:
    builder = (
        SparkSession.builder
        .appName("phase6-spark-streaming-kafka-to-parquet")
        .master(EFFECTIVE_SPARK_MASTER_URL)
        .config("spark.sql.shuffle.partitions", str(SPARK_SQL_SHUFFLE_PARTITIONS))
    )
    if SPARK_KAFKA_CONNECTOR_PACKAGE:
        builder = builder.config("spark.jars.packages", SPARK_KAFKA_CONNECTOR_PACKAGE)
    spark = builder.getOrCreate()
    spark.sparkContext.setLogLevel(os.getenv("LOG_LEVEL", "WARN"))
    print({"spark_version": spark.version, "spark_master": spark.sparkContext.master})
else:
    print("PySpark ist nicht verfügbar: Der gekennzeichnete pandas-Strukturtest-Fallback wird gewählt.")


{'spark_version': '4.1.2', 'spark_master': 'local[*]'}


#### Explizites Ereignisschema deklarieren

Structured Streaming leitet den JSON-Vertrag aus Phase 5 nicht automatisch ab. Nullable-Messfelder sind beabsichtigt, weil APIs vereinzelt Beobachtungen auslassen können.

In [4]:
EVENT_FIELDS = [
    "event_id", "schema_version", "source", "city_id", "event_time_utc",
    "ingestion_time_utc", "data_status", "pm2_5", "pm10", "no2",
]

if HAS_PYSPARK:
    event_schema = StructType([
        StructField("event_id", StringType(), nullable=True),
        StructField("schema_version", StringType(), nullable=True),
        StructField("source", StringType(), nullable=True),
        StructField("city_id", StringType(), nullable=True),
        StructField("event_time_utc", StringType(), nullable=True),
        StructField("ingestion_time_utc", StringType(), nullable=True),
        StructField("data_status", StringType(), nullable=True),
        StructField("pm2_5", DoubleType(), nullable=True),
        StructField("pm10", DoubleType(), nullable=True),
        StructField("no2", DoubleType(), nullable=True),
    ])
    print(f"OK: event_schema definiert — {len(event_schema.fields)} Felder, expliziter StructType für Spark from_json()")
else:
    print(f"OK: EVENT_FIELDS definiert — {len(EVENT_FIELDS)} Felder (PySpark nicht verfügbar, pandas-Fallback aktiv)")


OK: event_schema definiert — 10 Felder, expliziter StructType für Spark from_json()


#### Prüfen, ob Kafka-Einstellungen real sind

Platzhalter für Broker und Gruppen-Topic dürfen niemals einen echten Kafka-Verbindungsversuch auslösen.

In [5]:
def kafka_configured() -> bool:
    return "<" not in KAFKA_BOOTSTRAP_SERVERS and "gXX" not in KAFKA_TOPIC

print("OK: kafka_configured definiert")


OK: kafka_configured definiert


#### TCP-Erreichbarkeit des Brokers prüfen

Die kurze Prüfung entscheidet, ob der automatische Modus die Spark-Kafka-Initialisierung versucht oder den lokalen Stream-Fallback verwendet.

In [6]:
def broker_reachable(bootstrap_servers: str, timeout_seconds: float = 2.0) -> tuple:
    try:
        host, port = bootstrap_servers.rsplit(":", 1)
        with socket.create_connection((host, int(port)), timeout=timeout_seconds):
            return True, None
    except Exception as exc:
        return False, str(exc)

print("OK: broker_reachable definiert")


OK: broker_reachable definiert


#### Echten Spark-Kafka-Stream definieren

Kafka-Metadaten wie Topic, Partition, Offset und Zeitstempel bleiben zur Nachvollziehbarkeit an jedem Ereignis erhalten.

In [7]:
def kafka_raw_stream():
    return (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
        .option("subscribe", KAFKA_TOPIC)
        .option("startingOffsets", "earliest")
        .option("failOnDataLoss", "false")
        .load()
        .select(
            col("topic").alias("kafka_topic"),
            col("partition"),
            col("offset"),
            col("timestamp").alias("kafka_timestamp"),
            col("key").cast("string").alias("kafka_key"),
            col("value").cast("string").alias("raw_json"),
        )
    )

print("OK: kafka_raw_stream definiert")


OK: kafka_raw_stream definiert


#### Lokalen Spark-Dateistream-Fallback definieren

Der Fallback kopiert die JSONL-Datei aus Phase 5 in einen isolierten Eingabeordner und liest sie mit Spark-Structured-Streaming. Er validiert Transformationen lokal, ohne einen Kafka-Nachweis zu behaupten.

In [8]:
def mock_raw_stream():
    source_path = PHASE5_EVENTS_PATH if PHASE5_EVENTS_PATH.exists() else PHASE5_SAMPLE_EVENTS_PATH
    assert source_path.exists(), (
        "Lokale JSONL-Ereignisse aus Phase 5 fehlen. Notebook 05 vor Notebook 06 ausführen."
    )
    for path in [
        MOCK_INPUT_DIR,
        BRONZE_STREAM_PATH,
        REJECTS_STREAM_PATH,
        SILVER_STREAM_PATH,
        CHECKPOINT_BRONZE,
        CHECKPOINT_REJECTS,
        CHECKPOINT_SILVER,
    ]:
        shutil.rmtree(path, ignore_errors=True)
    MOCK_INPUT_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(source_path, MOCK_INPUT_DIR / "phase5_events.jsonl")
    return (
        spark.readStream.text(str(MOCK_INPUT_DIR))
        .select(
            lit(f"mock:{KAFKA_TOPIC}").alias("kafka_topic"),
            lit(-1).cast("int").alias("partition"),
            lit(-1).cast("long").alias("offset"),
            current_timestamp().alias("kafka_timestamp"),
            lit(None).cast("string").alias("kafka_key"),
            col("value").alias("raw_json"),
        )
    )

print("OK: mock_raw_stream definiert")

OK: mock_raw_stream definiert


#### Streaming-Quelle auswählen

Der strikte Kafka-Modus bricht bei Broker- oder Konnektorproblemen ab. Der automatische Modus darf auf den expliziten Spark-Dateistream-Fallback wechseln, wenn dies erlaubt ist.

In [9]:
source_mode = "pandas_mock_no_pyspark" if not HAS_PYSPARK else "mock"
fallback_reason = "Das pyspark-Paket ist lokal nicht installiert." if not HAS_PYSPARK else None
if not HAS_PYSPARK and (SPARK_KAFKA_MODE == "kafka" or not ALLOW_SPARK_KAFKA_MOCK_FALLBACK):
    raise RuntimeError(f"Strikter Spark-Modus fehlgeschlagen: {fallback_reason}")

if HAS_PYSPARK and SPARK_KAFKA_MODE in {"auto", "kafka"}:
    if not kafka_configured():
        fallback_reason = "Kafka-Broker oder gruppenspezifisches Topic enthält noch einen Platzhalter."
    else:
        reachable, broker_error = broker_reachable(KAFKA_BOOTSTRAP_SERVERS)
        if not reachable:
            fallback_reason = f"TCP-Prüfung des Kafka-Brokers fehlgeschlagen: {broker_error}"
        else:
            try:
                raw_stream = kafka_raw_stream()
                source_mode = "kafka"
            except Exception as exc:
                connector_hint = ""
                if "Failed to find data source: kafka" in str(exc):
                    connector_hint = (
                        " Der Spark-Kafka-Connector fehlt im aktuellen Kernel. "
                        "Lokal fuer den strikten Docker-Nachweis http://localhost:8888 verwenden. "
                        "Bei einem Host-Entwicklungslauf SPARK_KAFKA_MODE=auto und "
                        "ALLOW_SPARK_KAFKA_MOCK_FALLBACK=true setzen."
                    )
                fallback_reason = f"Initialisierung der Spark-Kafka-Quelle fehlgeschlagen: {exc}{connector_hint}"

if HAS_PYSPARK and source_mode != "kafka":
    if SPARK_KAFKA_MODE == "kafka" or not ALLOW_SPARK_KAFKA_MOCK_FALLBACK:
        raise RuntimeError(f"Strikter Kafka-Modus fehlgeschlagen: {fallback_reason}")
    try:
        raw_stream = mock_raw_stream()
    except Exception as _mock_exc:
        _exc_str = str(_mock_exc)
        if "getSubject" in _exc_str or "UnsupportedOperationException" in _exc_str:
            fallback_reason = (
                f"Spark/Java-Inkompatibilität (Java 21+, Subject.getSubject entfernt), "
                f"pandas-Fallback aktiv: {_exc_str[:200]}"
            )
            try:
                spark.stop()
            except Exception:
                pass
            HAS_PYSPARK = False
            source_mode = "pandas_mock_no_pyspark"
        else:
            raise

if HAS_PYSPARK:
    print({"spark_version": spark.version, "spark_master": spark.sparkContext.master,
           "selected_source_mode": source_mode, "fallback_reason": fallback_reason})
    event_schema
else:
    print({"selected_source_mode": source_mode, "fallback_reason": fallback_reason})

{'selected_source_mode': 'pandas_mock_no_pyspark', 'fallback_reason': 'Spark/Java-Inkompatibilität (Java 21+, Subject.getSubject entfernt), pandas-Fallback aktiv: getSubject is not supported'}


### Lokaler Strukturtest-Fallback ohne PySpark

Falls PySpark lokal nicht installiert ist, validiert dieser ausdrücklich gekennzeichnete Fallback die Datenstruktur mit pandas. Er weist weder eine Spark-Ausführung noch das Einlesen aus Kafka nach.

#### Lokale JSONL-Ereignisse laden und normalisieren

Die JSONL-Ereignisse aus Phase 5 werden eingelesen und in eine tabellarische Struktur überführt.

In [10]:
if not HAS_PYSPARK:
    import json
    import pandas as pd

    source_path = PHASE5_EVENTS_PATH if PHASE5_EVENTS_PATH.exists() else PHASE5_SAMPLE_EVENTS_PATH
    assert source_path.exists(), "JSONL-Ereignisse aus Phase 5 fehlen. Notebook 05 vor Notebook 06 ausführen."
    raw_lines = [line for line in source_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    records = [json.loads(line) for line in raw_lines]
    pandas_bronze_df = pd.DataFrame(records).reindex(columns=EVENT_FIELDS)
    pandas_bronze_df["raw_json"] = raw_lines
    pandas_bronze_df["event_time_ts"] = pd.to_datetime(pandas_bronze_df["event_time_utc"], utc=True, errors="coerce")
    pandas_bronze_df["ingestion_time_ts"] = pd.to_datetime(pandas_bronze_df["ingestion_time_utc"], utc=True, errors="coerce")
    print(f"OK (pandas-Fallback): {len(pandas_bronze_df)} Bronze-Ereignisse aus {source_path.name} geladen")


OK (pandas-Fallback): 8 Bronze-Ereignisse aus open_meteo_air_quality_events.jsonl geladen


#### Qualitätsregeln und Stadtanreicherung anwenden

Pflichtfelder, Wertebereiche und bekannte Städte werden geprüft. Ungültige Zeilen bleiben als Rejects nachvollziehbar.

In [11]:
if not HAS_PYSPARK:
    required = ["event_id", "schema_version", "source", "city_id", "event_time_utc", "ingestion_time_utc"]
    required_valid = pandas_bronze_df[required].notna().all(axis=1)
    schema_valid = pandas_bronze_df["schema_version"].eq("1.0") & pandas_bronze_df["source"].eq("open_meteo")
    plausible = (
        (pandas_bronze_df["pm2_5"].isna() | pandas_bronze_df["pm2_5"].between(0, 1000, inclusive="both"))
        & (pandas_bronze_df["pm10"].isna() | pandas_bronze_df["pm10"].between(0, 2000, inclusive="both"))
        & (pandas_bronze_df["no2"].isna() | pandas_bronze_df["no2"].between(0, 1000, inclusive="both"))
    )
    timestamps_valid = pandas_bronze_df[["event_time_ts", "ingestion_time_ts"]].notna().all(axis=1)
    valid_mask = required_valid & schema_valid & plausible & timestamps_valid
    pandas_rejects_df = pandas_bronze_df.loc[~valid_mask].copy()
    pandas_rejects_df["reject_reason"] = "event_quality_validation_failed"
    valid_events_df = pandas_bronze_df.loc[valid_mask].drop_duplicates("event_id").copy()
    city_ref_cols = ["city_id", "city_name", "country_code", "latitude", "longitude"]
    city_meta_cols = ["city_id", "population", "area_km2", "population_density", "parse_status"]
    city_reference_pd = pd.read_parquet(CITY_REFERENCE_PATH)[city_ref_cols]
    city_metadata_pd = pd.read_parquet(CITY_METADATA_PATH)[city_meta_cols]
    pandas_silver_df = valid_events_df.merge(city_reference_pd, on="city_id", how="left")
    pandas_silver_df = pandas_silver_df.merge(city_metadata_pd, on="city_id", how="left")
    print(f"OK (pandas-Fallback): {len(valid_events_df)} gültige / {len(pandas_rejects_df)} abgelehnte Ereignisse")


OK (pandas-Fallback): 8 gültige / 0 abgelehnte Ereignisse


#### Kompatible Parquet-Übergaben speichern

Der Strukturtest schreibt dieselben Ausgabeformate wie der Spark-Pfad, damit nachgelagerte Notebooks lokal geprüft werden können.

In [12]:
if not HAS_PYSPARK:
    def write_pandas_dataset(frame, output_dir: Path):
        if output_dir.is_dir():
            shutil.rmtree(output_dir)
        elif output_dir.exists():
            output_dir.unlink()
        output_dir.mkdir(parents=True, exist_ok=True)
        frame.to_parquet(output_dir / "part-00000.parquet", index=False)

    assert pandas_silver_df["city_name"].notna().all(), "Unbekannte Stadt-IDs sind in Silver gelangt."
    pandas_silver_df["processing_mode"] = "pandas_mock_no_pyspark"
    latest_pd = pandas_silver_df.sort_values(["event_time_ts", "ingestion_time_ts"])
    latest_pd = latest_pd.drop_duplicates("city_id", keep="last").copy()
    latest_pd["dataset_context"] = "open_meteo_live"
    write_pandas_dataset(pandas_bronze_df, BRONZE_STREAM_PATH)
    write_pandas_dataset(pandas_rejects_df, REJECTS_STREAM_PATH)
    write_pandas_dataset(pandas_silver_df, SILVER_STREAM_PATH)
    write_pandas_dataset(latest_pd, LATEST_SNAPSHOT_PATH)
    pandas_fallback_summary = {
        "selected_source_mode": source_mode, "bronze_row_count": len(pandas_bronze_df),
        "silver_row_count": len(pandas_silver_df), "reject_row_count": len(pandas_rejects_df),
        "latest_snapshot_row_count": len(latest_pd),
    }
    print(pandas_fallback_summary)

{'selected_source_mode': 'pandas_mock_no_pyspark', 'bronze_row_count': 8, 'silver_row_count': 8, 'reject_row_count': 0, 'latest_snapshot_row_count': 8}


### Ereignisse parsen, validieren und anreichern

Kafka-Metadaten bleiben erhalten. JSON wird mit einem expliziten Schema geparst. Ungültiges JSON, ungültige Felder und unbekannte Stadt-IDs werden als sichtbare Reject-Zeilen abgelegt.

#### Kafka- oder Mock-JSON-Werte parsen

Der rohe JSON-String wird in ein strukturiertes Ereignis umgewandelt. Ursprüngliche Metadaten bleiben für Rejects und Nachvollziehbarkeit verfügbar.

In [13]:
if HAS_PYSPARK:
    parsed_stream = (
        raw_stream
        .withColumn("event", from_json(col("raw_json"), event_schema))
    )

    json_valid_stream = (
        parsed_stream
        .filter(col("event").isNotNull())
        .select(
            "kafka_topic", "partition", "offset", "kafka_timestamp", "kafka_key", "raw_json",
            col("event.*"),
        )
    )
    print("OK: parsed_stream und json_valid_stream definiert (from_json mit explizitem StructType)")


#### Qualitätsregeln für Ereignisse definieren

Pflichtfelder, Schemaversion, Quelle, Zeitstempel und plausible Schadstoffbereiche bilden wiederverwendbare Validierungsregeln.

In [14]:
if HAS_PYSPARK:
    required_fields_valid = (
        col("event_id").isNotNull()
        & col("schema_version").isNotNull()
        & col("source").isNotNull()
        & col("city_id").isNotNull()
        & col("event_time_utc").isNotNull()
        & col("ingestion_time_utc").isNotNull()
    )
    schema_valid = (col("schema_version") == lit("1.0")) & (col("source") == lit("open_meteo"))
    pollutants_plausible = (
        (col("pm2_5").isNull() | col("pm2_5").between(0, 1000))
        & (col("pm10").isNull() | col("pm10").between(0, 2000))
        & (col("no2").isNull() | col("no2").between(0, 1000))
    )
    print("OK: Qualitätsregeln definiert — required_fields_valid, schema_valid, pollutants_plausible")


#### Gültige Ereignisse und Rejects trennen

Zeitstempel werden geparst und deterministische Ereignis-IDs vor der Anreicherung dedupliziert.

In [15]:
if HAS_PYSPARK:
    quality_marked_stream = (
        json_valid_stream
        .withColumn("event_time_ts", to_timestamp("event_time_utc"))
        .withColumn("ingestion_time_ts", to_timestamp("ingestion_time_utc"))
        .withColumn("is_required_fields_valid", required_fields_valid)
        .withColumn("is_schema_valid", schema_valid)
        .withColumn("is_pollutant_range_plausible", pollutants_plausible)
        .withColumn(
            "is_event_valid",
            col("is_required_fields_valid")
            & col("is_schema_valid")
            & col("is_pollutant_range_plausible")
            & col("event_time_ts").isNotNull()
            & col("ingestion_time_ts").isNotNull(),
        )
    )

    quality_valid_stream = quality_marked_stream.filter(col("is_event_valid")).dropDuplicates(["event_id"])
    quality_rejects_stream = (
        quality_marked_stream.filter(~col("is_event_valid"))
        .withColumn("reject_reason", lit("event_quality_validation_failed"))
    )
    json_rejects_stream = (
        parsed_stream.filter(col("event").isNull())
        .select("kafka_topic", "partition", "offset", "kafka_timestamp", "kafka_key", "raw_json")
        .withColumn("reject_reason", lit("invalid_json"))
    )
    print("OK: quality_marked_stream, quality_valid_stream, quality_rejects_stream, json_rejects_stream definiert")


#### Statische Stadtdatensätze laden

Die Stream-Verknüpfung benötigt die Städtereferenz aus Phase 2 und die Wikipedia-Metadaten aus Phase 4. Erwartete Spalten werden vor der Verwendung geprüft.

In [16]:
if HAS_PYSPARK:
    city_reference_df = spark.read.parquet(str(CITY_REFERENCE_PATH))
    city_metadata_df = spark.read.parquet(str(CITY_METADATA_PATH))
    required_city_ref_cols = {"city_id", "city_name", "country_code", "latitude", "longitude"}
    required_city_meta_cols = {"city_id", "population", "area_km2", "population_density", "parse_status"}
    assert not (required_city_ref_cols - set(city_reference_df.columns)), "In city_reference fehlen erforderliche Spalten"
    assert not (required_city_meta_cols - set(city_metadata_df.columns)), "In city_metadata fehlen erforderliche Spalten"
    print(f"OK: city_reference ({len(city_reference_df.columns)} Spalten) und city_metadata ({len(city_metadata_df.columns)} Spalten) geladen — Pflichtfelder vorhanden")


#### Ereignisse anreichern und unbekannte Städte isolieren

Left Joins halten unbekannte `city_id`-Werte sichtbar, damit sie in den Reject-Datensatz geleitet werden können.

In [17]:
if HAS_PYSPARK:
    enriched_stream = (
        quality_valid_stream
        .join(city_reference_df.select(*sorted(required_city_ref_cols)), on="city_id", how="left")
        .join(city_metadata_df.select(*sorted(required_city_meta_cols)), on="city_id", how="left")
    )
    known_city_stream = enriched_stream.filter(col("city_name").isNotNull())
    unknown_city_stream = (
        enriched_stream.filter(col("city_name").isNull())
        .withColumn("reject_reason", lit("unknown_city_id"))
    )
    print("OK: enriched_stream, known_city_stream, unknown_city_stream definiert (Left-Join auf city_reference)")


#### Reject-Gründe zusammenführen

Ungültiges JSON, fehlgeschlagene Qualitätsregeln und unbekannte Städte werden in einem Reject-Stream vereinheitlicht.

In [18]:
if HAS_PYSPARK:
    reject_columns = ["kafka_topic", "partition", "offset", "kafka_timestamp", "kafka_key", "raw_json", "reject_reason"]
    rejects_stream = (
        json_rejects_stream.select(*reject_columns)
        .unionByName(quality_rejects_stream.select(*reject_columns))
        .unionByName(unknown_city_stream.select(*reject_columns))
    )

    print("Explizites Ereignisschema:")
    event_schema

### Bronze, Silver, Rejects und neuesten Snapshot schreiben

Endliche Trigger halten das Notebook reproduzierbar. Jeder Ausgabeweg besitzt einen eigenen Checkpoint. Im Mock-Modus werden vor der Ausführung nur Mock-spezifische Ausgaben und Checkpoints zurückgesetzt.

#### Endlichen Parquet-Streaming-Writer definieren

Notebook-Läufe sollen terminieren. Der Helfer bevorzugt `availableNow` und verwendet bei älteren Spark-Laufzeiten ersatzweise einen einmaligen Trigger.

In [19]:
if HAS_PYSPARK:
    def start_finite_parquet_query(stream_df, output_path: Path, checkpoint_path: Path):
        writer = (
            stream_df.writeStream
            .format("parquet")
            .option("path", str(output_path))
            .option("checkpointLocation", str(checkpoint_path))
        )
        try:
            return writer.trigger(availableNow=True).start()
        except TypeError:
            return writer.trigger(once=True).start()

    print("OK: start_finite_parquet_query definiert (trigger=availableNow mit Fallback auf once)")


#### Bronze-, Reject- und Silver-Abfragen ausführen

Jede Ausgabe erhält einen eigenen Checkpoint. Abfragefehler werden unmittelbar nach der Terminierung ausgelöst.

In [20]:
if HAS_PYSPARK:
    queries = [
        start_finite_parquet_query(json_valid_stream, BRONZE_STREAM_PATH, CHECKPOINT_BRONZE),
        start_finite_parquet_query(rejects_stream, REJECTS_STREAM_PATH, CHECKPOINT_REJECTS),
        start_finite_parquet_query(known_city_stream, SILVER_STREAM_PATH, CHECKPOINT_SILVER),
    ]
    for query in queries:
        query.awaitTermination()
        if query.exception():
            raise RuntimeError(str(query.exception()))
    print(f"OK: alle 3 Streaming-Queries abgeschlossen — Bronze: {BRONZE_STREAM_PATH.name}, Rejects: {REJECTS_STREAM_PATH.name}, Silver: {SILVER_STREAM_PATH.name}")


#### Gespeicherte Streaming-Ausgaben erneut lesen

Das erneute Einlesen macht die Persistenz zu einem expliziten Test, statt eine verwendbare Parquet-Ausgabe nur anzunehmen.

In [21]:
if HAS_PYSPARK:
    def has_parquet_files(path) -> bool:
        return Path(path).exists() and any(Path(path).rglob("*.parquet"))

    assert has_parquet_files(BRONZE_STREAM_PATH), (
        f"Keine Bronze-Parquet-Dateien geschrieben: {BRONZE_STREAM_PATH}. "
        "Kafka-Events veröffentlichen (Notebook 05) oder Checkpoint löschen."
    )
    assert has_parquet_files(SILVER_STREAM_PATH), (
        f"Keine Silver-Parquet-Dateien geschrieben: {SILVER_STREAM_PATH}. "
        "Prüfen ob gültige Events mit bekannten city_ids vorhanden sind."
    )
    bronze_readback_df = spark.read.parquet(str(BRONZE_STREAM_PATH))
    silver_readback_df = spark.read.parquet(str(SILVER_STREAM_PATH))
    reject_count = (
        spark.read.parquet(str(REJECTS_STREAM_PATH)).count()
        if has_parquet_files(REJECTS_STREAM_PATH) else 0
    )
    print(f"OK: Readback — Bronze: {bronze_readback_df.count()} Zeilen, Silver: {silver_readback_df.count()} Zeilen, Rejects: {reject_count}")


#### Neuesten Live-Snapshot erstellen

Ein Fenster wählt je Stadt das neueste Ereignis und schreibt eine kompakte Gold-Übergabe für Phase 7.

In [22]:
if HAS_PYSPARK:
    latest_window = Window.partitionBy("city_id").orderBy(desc("event_time_ts"), desc("ingestion_time_ts"))
    latest_snapshot_df = (
        silver_readback_df
        .withColumn("row_number", row_number().over(latest_window))
        .filter(col("row_number") == 1)
        .drop("row_number")
        .withColumn("dataset_context", lit("open_meteo_live"))
    )
    LATEST_SNAPSHOT_PATH.parent.mkdir(parents=True, exist_ok=True)
    latest_snapshot_df.write.mode("overwrite").parquet(str(LATEST_SNAPSHOT_PATH))
    latest_readback_df = spark.read.parquet(str(LATEST_SNAPSHOT_PATH))
    print(f"OK: Live-Snapshot — {latest_readback_df.count()} Städte → {LATEST_SNAPSHOT_PATH}")


#### Verarbeitungszusammenfassung anzeigen

Anzahlen, Schema und Beispielzeilen machen das technische Ergebnis im Notebook nachvollziehbar.

In [23]:
if HAS_PYSPARK:
    result_summary = {
        "selected_source_mode": source_mode,
        "fallback_reason": fallback_reason,
        "bronze_row_count": bronze_readback_df.count(),
        "silver_row_count": silver_readback_df.count(),
        "reject_row_count": reject_count,
        "latest_snapshot_row_count": latest_readback_df.count(),
    }
    print(result_summary)
    silver_readback_df.printSchema()
    silver_readback_df.select("city_id", "city_name", "event_time_ts", "data_status", "pm2_5", "pm10", "no2").show(10, truncate=False)

## Validierung und Qualitätsprüfungen

Die Assertions prüfen Spark-Ausführung, erneutes Einlesen aus Parquet, Join-Abdeckung, Deduplizierung, plausible Schadstoffbereiche, Kardinalität des neuesten Snapshots und den Ausgabeort. Ein Mock-Lauf darf nie als Kafka-Nachweis ausgewiesen werden.

In [24]:
if HAS_PYSPARK:
    silver_count = silver_readback_df.count()
    bronze_count = bronze_readback_df.count()
    latest_count = latest_readback_df.count()
    duplicate_event_ids = silver_readback_df.groupBy("event_id").count().filter(col("count") > 1).count()
    unknown_city_count = silver_readback_df.filter(col("city_name").isNull()).count()
    invalid_pollutant_count = silver_readback_df.filter(
        ~(col("pm2_5").isNull() | col("pm2_5").between(0, 1000))
        | ~(col("pm10").isNull() | col("pm10").between(0, 2000))
        | ~(col("no2").isNull() | col("no2").between(0, 1000))
    ).count()

    assert bronze_count > 0, "Keine Bronze-Zeilen geschrieben. Kafka-Ereignisse veröffentlichen oder Notebook 05 erneut für Mock-Eingaben ausführen."
    assert silver_count > 0, "Keine angereicherten Silver-Zeilen geschrieben."
    assert duplicate_event_ids == 0, f"Doppelte Silver-Ereignis-IDs gefunden: {duplicate_event_ids}"
    assert unknown_city_count == 0, f"Zeilen mit unbekannter Stadt sind in Silver gelangt: {unknown_city_count}"
    assert invalid_pollutant_count == 0, f"Zeilen mit unplausiblen Schadstoffwerten sind in Silver gelangt: {invalid_pollutant_count}"
    assert latest_count <= city_reference_df.select("city_id").distinct().count(), "Der neueste Snapshot enthält zu viele Zeilen"
    assert not (PROJECT_ROOT / "notebooks" / "data").exists(), "Ausgaben wurden unter notebooks/data geschrieben"

    quality_summary_df = silver_readback_df.groupBy("data_status").count().orderBy("data_status")
    quality_summary_df.show(truncate=False)
    print({
        "spark_read_kafka_requirement_proven": source_mode == "kafka",
        "local_spark_streaming_fallback_tested": source_mode == "mock",
        "silver_rows": silver_count,
        "latest_snapshot_rows": latest_count,
        "reject_rows": reject_count,
    })

    spark.stop()

#### Reinen pandas-Strukturtest validieren

Wenn der lokale Fallback ohne PySpark aktiv ist, werden Parquet-Ausgaben, Rejects, Deduplizierung und Stadtanreicherung ausdrücklich geprüft.

In [25]:
if not HAS_PYSPARK:
    assert len(pandas_bronze_df) > 0, "Keine Bronze-Zeilen geschrieben."
    assert len(pandas_silver_df) > 0, "Keine angereicherten Silver-Zeilen geschrieben."
    assert pandas_silver_df["event_id"].duplicated().sum() == 0, "Doppelte Silver-Ereignis-IDs gefunden."
    assert pandas_silver_df["city_name"].notna().all(), "Zeilen mit unbekannter Stadt sind in Silver gelangt."
    assert len(latest_pd) <= city_reference_pd["city_id"].nunique(), "Der neueste Snapshot enthält zu viele Zeilen."
    assert not (PROJECT_ROOT / "notebooks" / "data").exists(), "Ausgaben wurden unter notebooks/data geschrieben."
    print({
        "spark_read_kafka_requirement_proven": False,
        "local_pandas_structure_fallback_tested": True,
        "silver_rows": len(pandas_silver_df),
        "latest_snapshot_rows": len(latest_pd),
        "reject_rows": len(pandas_rejects_df),
    })

{'spark_read_kafka_requirement_proven': False, 'local_pandas_structure_fallback_tested': True, 'silver_rows': 8, 'latest_snapshot_rows': 8, 'reject_rows': 0}


## Ergebnisse

Das Notebook gibt `result_summary` und ein abschließendes Validierungswörterbuch aus. Für den verbindlichen FH-Nachweis müssen `selected_source_mode=kafka` und `spark_read_kafka_requirement_proven=True` gelten.

Für die lokale Entwicklung ist `selected_source_mode=mock` zu erwarten, wenn Broker oder Spark-Kafka-Konnektor nicht erreichbar sind. Dabei werden Spark-Structured-Streaming, explizites JSON-Parsing, Validierung, Joins, Checkpoints, Parquet-Schreibvorgänge und erneutes Einlesen dennoch ausgeführt.

## Einschränkungen

- Ein lokaler Mock-Lauf ist kein Kafka-Nachweis.
- Entfernte Spark-Worker benötigen gemeinsamen Zugriff auf `DATA_DIR` und `CHECKPOINT_DIR`.
- Kafka liefert mindestens einmal aus; deterministische `event_id`-Werte und Spark-Deduplizierung reduzieren Duplikate.
- Open-Meteo-Zeilen mit `controlled_offline_fallback` weisen nur die Mechanik nach und dürfen keine analytischen Aussagen stützen.

## Nächster Schritt
Notebook `07_gold_layer_and_data_quality.ipynb` ausführen, um aus historischen EEA-Silver-Daten und dem getrennten Live-Snapshot die analysefertige Gold-Schicht zu erstellen.